# FreshLens FL-2TC: Two-Tier Produce Classifier Training Pipeline

This Google Colab notebook trains and evaluates the complete **FreshLens Two-Tier Classifier (FL-2TC)** using GPU acceleration:
- **Tier 1 (Identity)**: Classifies `banana`, `cucumber`, `eggplant`, `tomato`, and rejects unsupported objects via `unknown`.
- **Tier 2 (Freshness)**: 3-stage freshness categorization (`fresh`, `medium`, `spoiled`) based on **Fahad et al. (CMC 2022)**.
- **Dataset**: Hugging Face [`SnapStock-AI/snapstock-freshness-dataset-v2`](https://huggingface.co/datasets/SnapStock-AI/snapstock-freshness-dataset-v2).
- **Architecture**: Ultralytics **YOLO11s-cls** with cosine annealing learning rate and texture-preserving augmentations.

## 1. Verify GPU Acceleration

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name:     {torch.cuda.get_device_name(0)}")
else:
    print("WARNING: Running on CPU! Go to Runtime -> Change runtime type -> T4 GPU or A100.")

## 2. Install Required Dependencies

In [ ]:
!pip install -q ultralytics huggingface_hub datasets

## 3. Clone FreshLens-AI Repository

In [ ]:
import os
from pathlib import Path

WORKSPACE_DIR = Path("/content/FreshLens-AI")
if not WORKSPACE_DIR.exists():
    !git clone https://github.com/FreshLens-AI/FreshLens-AI.git /content/FreshLens-AI

%cd /content/FreshLens-AI/packages/ml

## 4. Authenticate with Hugging Face

The dataset `SnapStock-AI/snapstock-freshness-dataset-v2` requires an authorized Hugging Face token.
You can set `HF_TOKEN` in Colab **Secrets** (key icon on left sidebar) or enter it interactively below.

In [ ]:
import os
import getpass
from huggingface_hub import login

hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get('HF_TOKEN')
except Exception:
    pass

if not hf_token:
    hf_token = os.environ.get('HF_TOKEN') or getpass.getpass("Enter your Hugging Face Access Token: ")

os.environ["HF_TOKEN"] = hf_token
login(token=hf_token)
print("Hugging Face authentication successful!")

## 5. Download & Prepare Datasets

Downloads the SnapStock freshness dataset and partitions it into stratified, deduplicated train/val/test splits for both Model 1 and Model 2.

In [ ]:
!python -m training.dataset_snapstock \
  --repo-id SnapStock-AI/snapstock-freshness-dataset-v2 \
  --output-dir /content/data/fl2tc \
  --seed 21

## 6. Train Tier 1: Identity Classifier

Trains `yolo11s-cls` on `banana`, `cucumber`, `eggplant`, `tomato`, and `unknown`.

In [ ]:
!python -m training.train_identity \
  --data /content/data/fl2tc/identity \
  --model yolo11s-cls.pt \
  --epochs 80 \
  --imgsz 224 \
  --batch 64 \
  --device 0 \
  --project /content/runs/identity \
  --name identity-yolo11s-cls-v2

## 7. Train Tier 2: Freshness Classifier

Trains `yolo11s-cls` on 3 freshness stages (`fresh`, `medium`, `spoiled`) with color-preserving augmentations.

In [ ]:
!python -m training.train_freshness \
  --data /content/data/fl2tc/freshness \
  --model yolo11s-cls.pt \
  --epochs 70 \
  --imgsz 224 \
  --batch 64 \
  --device 0 \
  --project /content/runs/freshness \
  --name freshness-yolo11s-cls-v2

## 8. Comprehensive Evaluation & Quality Reports

Evaluates both models on held-out test splits, generating precision, recall, F1, latency, and severe error metrics.

In [ ]:
!python -m training.evaluate_fl2tc \
  --identity-weights /content/runs/identity/identity-yolo11s-cls-v2/weights/best.pt \
  --freshness-weights /content/runs/freshness/freshness-yolo11s-cls-v2/weights/best.pt \
  --dataset-dir /content/data/fl2tc \
  --split test \
  --device cpu \
  --output-dir /content/runs/eval

# Display summary
import json
from pathlib import Path
summary = json.loads(Path("/content/runs/eval/fl2tc-test-summary.json").read_text())
print(json.dumps(summary, indent=2))

## 9. Package & Download Artifacts

Packages the trained checkpoints and metrics files into a downloadable archive for deployment in FreshLens API & ML Worker.

In [ ]:
import shutil
from google.colab import files

RELEASE_DIR = Path("/content/freshlens_fl2tc_release")
RELEASE_DIR.mkdir(parents=True, exist_ok=True)

# Copy model weights
shutil.copy2("/content/runs/identity/identity-yolo11s-cls-v2/weights/best.pt", RELEASE_DIR / "identity-v2.pt")
shutil.copy2("/content/runs/freshness/freshness-yolo11s-cls-v2/weights/best.pt", RELEASE_DIR / "freshness-v2.pt")

# Copy metrics
for p in Path("/content/runs/eval").glob("*.json"):
    shutil.copy2(p, RELEASE_DIR / p.name)

# Create zip
archive_path = shutil.make_archive("/content/freshlens-fl2tc-v2-artifacts", "zip", RELEASE_DIR)
print(f"Archive created: {archive_path}")

# Download to local computer
files.download(archive_path)